In [ ]:
# ===========================================
# Setup: Clone repo and build frontend
# ===========================================

# Clone repository
!git clone https://github.com/unslothai/new-ui-prototype.git
%cd /content/new-ui-prototype

# Install Node.js 20.x
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
print("✅ Node.js installed")

# Build frontend
%cd studio/frontend
!npm install --silent
!npm run build
print("✅ Frontend built")

# Install Python dependencies
%cd /content/new-ui-prototype
!pip install -q unsloth
!pip install -q -r studio/backend/requirements.txt
!pip install -q huggingface_hub datasets python-jose[cryptography] passlib[bcrypt] python-multipart matplotlib pandas ujson
print("✅ Python dependencies installed")

In [ ]:
# ===========================================
# Start Unsloth Studio
# ===========================================
import sys
sys.path.insert(0, '/content/new-ui-prototype/studio/backend')

from colab import start
start()

<div align="center">
  <img src="https://raw.githubusercontent.com/unslothai/unsloth/main/images/unsloth%20logo%20white%20text.png" width="400"/>
</div>

# 🦥 Unsloth Studio on Google Colab

A modern, full-stack web interface for fine-tuning, managing, and chatting with large language models.

**Features:**
- 🎯 **Training**: LoRA/QLoRA fine-tuning with real-time progress streaming
- 🤖 **Model Management**: Browse and load Hugging Face models
- 💬 **Inference**: Interactive chat playground
- 📊 **Dataset Tools**: Upload and preview datasets
- 🚀 **Export**: Push trained adapters to Hugging Face Hub

---

**⚠️ Important Notes:**
- **Private Repository**: You'll need a GitHub Personal Access Token to clone
- Use a **GPU runtime** for training (Runtime → Change runtime type → T4 GPU)
- The notebook will expose the UI via **Cloudflare Tunnel** (no account needed)
- Your first launch will generate a **setup token** for creating an admin account

---

**Repository**: [github.com/unslothai/new-ui-prototype](https://github.com/unslothai/new-ui-prototype/tree/nightly)

## 📋 Step 1: Install System Dependencies

Install Node.js and required system packages.

## 🔐 Step 1.5: Authenticate with GitHub (Private Repo)

Since this is a private repository, you need to authenticate with GitHub. 

**Get a Personal Access Token (classic):**
1. Go to https://github.com/settings/tokens
2. Click "Generate new token (classic)"
3. Give it `repo` scope
4. Copy the token

**Or use GitHub CLI:**

In [ ]:
import os
from getpass import getpass

# Option 1: Use GitHub Personal Access Token
print("🔐 GitHub Authentication Required (Private Repo)")
print("=" * 60)
print("Get a token from: https://github.com/settings/tokens")
print("Required scope: 'repo'")
print("=" * 60)

github_token = getpass("Enter your GitHub Personal Access Token: ")

if github_token:
    # Store token for git operations
    os.environ['GITHUB_TOKEN'] = github_token
    print("✅ Token stored (will be used for cloning)")
else:
    print("⚠️ No token provided - clone may fail for private repo")

In [ ]:
%%bash
# Install Node.js 20.x
echo "📦 Installing Node.js..."
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs

# Verify installation
echo "✅ Node.js $(node -v) installed"
echo "✅ npm $(npm -v) installed"

## 📥 Step 2: Clone Repository and Install Python Dependencies

Clone the repository and install Unsloth + backend dependencies.

In [ ]:
import os
from pathlib import Path

# Clone repository (use nightly branch)
repo_path = Path("/content/new-ui-prototype")
if not repo_path.exists():
    print("📥 Cloning repository...")
    
    # Use token if available (for private repo)
    github_token = os.environ.get('GITHUB_TOKEN', '')
    if github_token:
        # Clone with token embedded in URL
        repo_url = f"https://{github_token}@github.com/unslothai/new-ui-prototype.git"
        !git clone -b nightly {repo_url}
    else:
        # Try without token (will work for public repo)
        !git clone -b nightly https://github.com/unslothai/new-ui-prototype.git
else:
    print("✅ Repository already cloned")

# Change to repo directory
os.chdir(repo_path)
print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
%%bash
# Install Unsloth
echo "📦 Installing Unsloth..."
pip install --no-cache-dir unsloth

# Install backend dependencies
echo "📦 Installing backend dependencies..."
cd studio/backend
pip install --no-cache-dir -r requirements.txt

# Install additional required packages
pip install --no-cache-dir \
    huggingface_hub \
    datasets \
    python-jose[cryptography] \
    passlib[bcrypt] \
    python-multipart \
    matplotlib \
    pandas \
    ujson

echo "✅ Python dependencies installed"

## 🎨 Step 3: Build Frontend

Build the React/TypeScript frontend.

In [ ]:
%%bash
cd studio/frontend

echo "📦 Installing frontend dependencies..."
npm install --legacy-peer-deps

echo "🏗️ Building frontend..."
npm run build

echo "✅ Frontend built to studio/frontend/dist"

## 🌐 Step 4: Set Up Cloudflare Tunnel

Install cloudflared to expose the backend server to the internet.

In [ ]:
%%bash
# Install cloudflared
if ! command -v cloudflared &> /dev/null; then
    echo "📦 Installing Cloudflare Tunnel..."
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    sudo dpkg -i cloudflared-linux-amd64.deb
    rm cloudflared-linux-amd64.deb
    echo "✅ Cloudflared installed"
else
    echo "✅ Cloudflared already installed"
fi

cloudflared version

## 🚀 Step 5: Launch Unsloth Studio

Start the backend server and create a public tunnel.

**⚠️ IMPORTANT:**
1. Look for the **🔗 Public URL** in the output below
2. On first launch, look for the **🔑 Setup Token** (one-time use)
3. Open the URL in your browser
4. Use the setup token to create your admin account

**Note**: This cell will run continuously. To stop the server, click the ⏹️ stop button or interrupt the kernel.

In [ ]:
import subprocess
import time
import sys
import re
from threading import Thread
from queue import Queue, Empty

def stream_output(pipe, queue, prefix=""):
    """Stream subprocess output to queue"""
    for line in iter(pipe.readline, b''):
        queue.put((prefix, line.decode('utf-8')))
    pipe.close()

# Change to backend directory
os.chdir("/content/new-ui-prototype/studio/backend")

print("🚀 Starting Unsloth Studio...\n")
print("=" * 70)

# Start backend server
backend_process = subprocess.Popen(
    ["python", "run.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    bufsize=1
)

# Wait for server to start
print("⏳ Waiting for backend server to start...")
time.sleep(10)

# Start cloudflared tunnel
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    bufsize=1
)

# Create queues for output
backend_queue = Queue()
tunnel_queue = Queue()

# Start output streaming threads
Thread(target=stream_output, args=(backend_process.stdout, backend_queue, "[BACKEND]"), daemon=True).start()
Thread(target=stream_output, args=(backend_process.stderr, backend_queue, "[BACKEND]"), daemon=True).start()
Thread(target=stream_output, args=(tunnel_process.stdout, tunnel_queue, "[TUNNEL]"), daemon=True).start()
Thread(target=stream_output, args=(tunnel_process.stderr, tunnel_queue, "[TUNNEL]"), daemon=True).start()

# Monitor output and extract public URL and setup token
public_url = None
setup_token = None
url_pattern = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
token_pattern = re.compile(r'Setup token: ([a-f0-9-]+)')

print("\n📡 Monitoring server output...\n")
print("=" * 70)

try:
    while True:
        # Check backend output
        try:
            prefix, line = backend_queue.get(timeout=0.1)
            print(f"{prefix} {line}", end='')
            
            # Look for setup token
            if setup_token is None:
                token_match = token_pattern.search(line)
                if token_match:
                    setup_token = token_match.group(1)
                    print(f"\n{'=' * 70}")
                    print(f"🔑 SETUP TOKEN (save this!): {setup_token}")
                    print(f"{'=' * 70}\n")
        except Empty:
            pass
        
        # Check tunnel output
        try:
            prefix, line = tunnel_queue.get(timeout=0.1)
            print(f"{prefix} {line}", end='')
            
            # Look for public URL
            if public_url is None:
                url_match = url_pattern.search(line)
                if url_match:
                    public_url = url_match.group(0)
                    print(f"\n{'=' * 70}")
                    print(f"🔗 PUBLIC URL: {public_url}")
                    print(f"{'=' * 70}\n")
                    print("✅ Unsloth Studio is now accessible!\n")
                    if setup_token:
                        print(f"📝 Next steps:")
                        print(f"   1. Open: {public_url}")
                        print(f"   2. Use setup token: {setup_token}")
                        print(f"   3. Create your admin account\n")
                    print(f"{'=' * 70}\n")
        except Empty:
            pass
        
        # Check if processes are still running
        if backend_process.poll() is not None:
            print("\n❌ Backend server stopped unexpectedly")
            break
        if tunnel_process.poll() is not None:
            print("\n❌ Tunnel stopped unexpectedly")
            break
        
        time.sleep(0.1)

except KeyboardInterrupt:
    print("\n\n🛑 Shutting down...")
finally:
    backend_process.terminate()
    tunnel_process.terminate()
    backend_process.wait()
    tunnel_process.wait()
    print("✅ Server stopped")

## ⚙️ Optional: Configure Hugging Face Token

If you want to use gated models or push to Hugging Face Hub, set your token here.

In [ ]:
# Optional: Set your Hugging Face token
# Get your token from: https://huggingface.co/settings/tokens

from huggingface_hub import login

# Uncomment and set your token
# HF_TOKEN = "hf_..."
# login(token=HF_TOKEN)
# print("✅ Logged in to Hugging Face")

print("💡 To set your HF token, uncomment the code above or use the UI settings")

## 🔧 Troubleshooting

### Server won't start
- Make sure you're using a **GPU runtime** (Runtime → Change runtime type)
- Check that all previous cells completed successfully
- Try restarting the runtime and running all cells again

### Can't access the URL
- The Cloudflare tunnel URL is temporary and changes each time
- Make sure the server is still running (cell shows "🔗 PUBLIC URL")
- Try opening the URL in an incognito/private window

### Training fails
- Check that you have enough GPU memory for your model
- Try using 4-bit quantization (enabled by default)
- Use smaller batch sizes if you get OOM errors

### Lost setup token
- The token is shown once in the server output above
- If you need a new one, delete `/content/new-ui-prototype/studio/backend/auth.db` and restart

---

## 📚 Additional Resources

- **GitHub Repository**: https://github.com/unslothai/new-ui-prototype/tree/nightly
- **Unsloth Documentation**: https://github.com/unslothai/unsloth
- **API Documentation**: Access `/docs` on your running instance

---

## 💾 Saving Your Work

To save trained models and datasets:

1. **Mount Google Drive:**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for outputs
!mkdir -p /content/drive/MyDrive/unsloth_studio_outputs

2. **Copy outputs to Drive:**

In [ ]:
# Copy trained models to Google Drive
!cp -r /content/new-ui-prototype/outputs/* /content/drive/MyDrive/unsloth_studio_outputs/
print("✅ Outputs saved to Google Drive")

---

## 🎯 Quick Training Tips

### Recommended Settings for Colab Free (T4 GPU)

| Setting | Value | Note |
|---------|-------|------|
| Model | `unsloth/Qwen2.5-1.5B-Instruct` | Small but capable |
| Training Method | LoRA/QLoRA | Memory efficient |
| 4-bit Quantization | ✅ Enabled | Reduces memory usage |
| Batch Size | 2-4 | Depends on model size |
| Context Length | 2048 | Balance memory/performance |
| LoRA Rank | 16-64 | Higher = more capacity |
| Gradient Accumulation | 4 | Simulates larger batch |

### Sample Datasets (Quick Start)
- `mlabonne/FineTome-100k` - General instruction following
- `HuggingFaceH4/no_robots` - Clean chat conversations
- `vicgalle/alpaca-gpt4` - High-quality instructions

---

**🎉 Happy Fine-tuning!**